# IMPTC Blind-Zone Emergence Risk Prediction (Revised)
## 전처리 · 시각화 · 그래프 구성 — 완전판

**기존 노트북 대비 변경사항**
- Drive → `/tmp` 복사로 I/O 10× 속도 향상
- 전체 트랙(~5,133개) 로드 + 세션 그룹화 → 샘플 0개 문제 해결
- PM(scooter+cyclist) 집중 분석 (보행자 선택 제외)
- `src/` 모듈 통합: ObjectState/Frame/Scene → build_scene_graph()
- NetworkX 이종 그래프 시각화 추가

**데이터 구조**
```
imptc_trajectory/
  train/ eval/ test/
    {track_id}/
      track.json
        overview:    class_name, duration, first_ts, last_ts
        track_data:  {'1': {ts, coordinates[x,y,z], velocity}, ...}
```

In [ ]:
# ── 0-A. src/ 인라인 정의 (Colab 자체 완결) ──────────────────
# Drive에 데이터셋 폴더만 있으면 됩니다 — 프로젝트 파일 불필요

from __future__ import annotations
import math
from dataclasses import dataclass, field
from typing import Any

# ── dataset.py: 데이터 구조 ──────────────────────────────────────────
@dataclass
class ObjectState:
    object_id: str
    object_type: str = "unknown"
    x: float = 0.0
    y: float = 0.0
    vx: float = 0.0
    vy: float = 0.0
    heading: float = 0.0
    visible: bool = True
    raw: dict = field(default_factory=dict)

@dataclass
class Frame:
    frame_id: str
    timestamp: float = 0.0
    ego: ObjectState | None = None
    objects: list = field(default_factory=list)
    map_info: dict = field(default_factory=dict)
    raw: dict = field(default_factory=dict)

@dataclass
class Scene:
    scene_id: str
    frames: list = field(default_factory=list)
    source_path: str = ""
    raw: Any = None

# ── utils.py: 기하 유틸 ──────────────────────────────────────────────
def euclidean(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

def normalize_angle(angle):
    while angle > math.pi:  angle -= 2 * math.pi
    while angle < -math.pi: angle += 2 * math.pi
    return angle

def relative_angle(origin, heading, point):
    dx = point[0] - origin[0]; dy = point[1] - origin[1]
    return normalize_angle(math.atan2(dy, dx) - heading)

def object_type_flags(object_type):
    n = object_type.lower()
    is_occ = int(any(t in n for t in ["parked","vehicle","car","bus","truck","building","wall"]))
    is_vru = int(any(t in n for t in ["scooter","bicycle","bike","cyclist","pedestrian","walker"]))
    return is_occ, is_vru

def polygon_area(points):
    if len(points) < 3: return 0.0
    area = 0.0
    for i, (x1, y1) in enumerate(points):
        x2, y2 = points[(i + 1) % len(points)]
        area += x1 * y2 - x2 * y1
    return abs(area) / 2.0

def polygon_centroid(points):
    if not points: return (0.0, 0.0)
    sa, cx, cy = 0.0, 0.0, 0.0
    for i, (x1, y1) in enumerate(points):
        x2, y2 = points[(i + 1) % len(points)]
        cross = x1 * y2 - x2 * y1
        sa += cross; cx += (x1 + x2) * cross; cy += (y1 + y2) * cross
    sa *= 0.5
    if abs(sa) < 1e-9:
        return (sum(x for x, _ in points) / len(points), sum(y for _, y in points) / len(points))
    return (cx / (6.0 * sa), cy / (6.0 * sa))

def point_in_polygon(point, polygon):
    if len(polygon) < 3: return False
    x, y = point; inside = False; j = len(polygon) - 1
    for i, (xi, yi) in enumerate(polygon):
        xj, yj = polygon[j]
        if (yi > y) != (yj > y) and x < (xj - xi) * (y - yi) / max(yj - yi, 1e-12) + xi:
            inside = not inside
        j = i
    return inside

# ── label_builder.py: 라벨 생성 ──────────────────────────────────────
SCOOTER_TOKENS  = {"scooter", "e-scooter", "electric_scooter", "micromobility"}
OCCLUDER_TOKENS = {"parked", "vehicle", "car", "bus", "truck", "building", "wall", "occluder"}

def is_scooter(object_type):
    n = object_type.lower().replace(" ", "_")
    return any(t in n for t in SCOOTER_TOKENS)

def is_occluder_type(object_type):
    n = object_type.lower().replace(" ", "_")
    return any(t in n for t in OCCLUDER_TOKENS)

def build_risk_label(scene, time_window=3.0, distance_threshold=10.0):
    if not scene.frames: return 0
    first_seen = {}; previous_visible = set()
    for frame in scene.frames:
        ego_xy = (frame.ego.x, frame.ego.y) if frame.ego else (0.0, 0.0)
        occluders = [o for o in frame.objects if is_occluder_type(o.object_type)]
        for obj in frame.objects:
            if not is_scooter(obj.object_type) or not obj.visible: continue
            obj_xy = (obj.x, obj.y)
            appeared_now = obj.object_id not in previous_visible
            first_seen.setdefault(obj.object_id, frame.timestamp)
            near_ego = euclidean(ego_xy, obj_xy) <= distance_threshold
            near_occ = any(euclidean((o.x, o.y), obj_xy) <= distance_threshold for o in occluders)
            if (appeared_now or frame.timestamp - first_seen[obj.object_id] <= time_window) and near_ego and near_occ:
                return 1
        previous_visible = {o.object_id for o in frame.objects if o.visible}
    return 0

def build_frame_risk_label(frame, distance_threshold=10.0):
    ego_xy = (frame.ego.x, frame.ego.y) if frame.ego else (0.0, 0.0)
    occluders = [o for o in frame.objects if is_occluder_type(o.object_type)]
    for obj in frame.objects:
        if not is_scooter(obj.object_type) or not obj.visible: continue
        obj_xy = (obj.x, obj.y)
        if (euclidean(ego_xy, obj_xy) <= distance_threshold and
                any(euclidean((o.x, o.y), obj_xy) <= distance_threshold for o in occluders)):
            return 1
    return 0

def build_blind_zone_label(scene, frame_index, blind_zone, time_window=3.0, distance_threshold=6.0):
    start_time = scene.frames[frame_index].timestamp
    polygon = blind_zone.raw.get("polygon") if isinstance(blind_zone.raw, dict) else None
    for future in scene.frames[frame_index + 1:]:
        if future.timestamp - start_time > time_window: break
        for obj in future.objects:
            if is_scooter(obj.object_type) and obj.visible:
                obj_xy = (obj.x, obj.y)
                if polygon and point_in_polygon(obj_xy, polygon): return 1
                if euclidean((blind_zone.x, blind_zone.y), obj_xy) <= distance_threshold: return 1
    return 0

# ── graph_builder.py: 그래프 구성 ──────────────────────────────────────
OBJECT_TYPE_TO_ID = {
    "unknown":0,"ego_vehicle":1,"e_scooter":2,"scooter":2,"pedestrian":3,
    "cyclist":12,"vehicle":4,"car":4,"parked_vehicle":5,"bus":6,"truck":7,
    "crosswalk":8,"sidewalk":9,"lane":10,"occlusion_zone":11,
}
NODE_FEATURE_NAMES = [
    "x","y","vx","vy","heading","object_type_id","distance_to_reference",
    "relative_angle_to_reference","visibility","is_occluder","is_vulnerable_road_user",
    "speed","acceleration","blind_zone_area",
]
EDGE_FEATURE_NAMES = [
    "distance","relative_velocity_x","relative_velocity_y",
    "relative_heading","time_to_collision","visibility_blocked",
]

def type_id(object_type):
    return OBJECT_TYPE_TO_ID.get(object_type.lower().replace(" ", "_"), 0)

def node_features(node, ego):
    if ego is None: ego = ObjectState(object_id="ego", object_type="ego_vehicle")
    dist  = euclidean((ego.x, ego.y), (node.x, node.y))
    angle = relative_angle((ego.x, ego.y), ego.heading, (node.x, node.y))
    is_occ, is_vru = object_type_flags(node.object_type)
    spd  = node.raw.get("speed", math.hypot(node.vx, node.vy)) if isinstance(node.raw, dict) else math.hypot(node.vx, node.vy)
    acc  = math.hypot(node.raw.get("ax", 0.), node.raw.get("ay", 0.)) if isinstance(node.raw, dict) else 0.0
    area = node.raw.get("area", 0.0) if isinstance(node.raw, dict) else 0.0
    return [node.x, node.y, node.vx, node.vy, node.heading,
            float(type_id(node.object_type)), dist, angle,
            float(node.visible), float(is_occ), float(is_vru),
            float(spd), float(acc), float(area)]

def edge_features(src, dst, distance):
    rel_vx = dst.vx - src.vx; rel_vy = dst.vy - src.vy
    rel_h  = normalize_angle(dst.heading - src.heading)
    closing = -((dst.x - src.x)*rel_vx + (dst.y - src.y)*rel_vy) / max(distance, 1e-6)
    ttc = distance / max(closing, 1e-6) if closing > 0 else 999.0
    vis_blocked = float((not dst.visible) or (object_type_flags(src.object_type)[0] and distance < 10.0))
    return [distance, rel_vx, rel_vy, rel_h, min(ttc, 999.0), vis_blocked]

def infer_edge_type(src, dst, distance):
    if src.object_type == "occlusion_zone" or dst.object_type == "occlusion_zone":
        return "blind_zone_relation"
    if object_type_flags(src.object_type)[0] and object_type_flags(dst.object_type)[1] and distance < 10.0:
        return "occludes"
    if distance < 5.0:
        return "potential_conflict"
    return "spatial_near"

def compute_blind_zone_polygon(ego_xy, occluder_xy, occluder_radius=2.5, depth_m=25.0):
    ex, ey = ego_xy; ox, oy = occluder_xy
    dist = math.hypot(ox - ex, oy - ey)
    if dist <= occluder_radius: return None
    ac = math.atan2(oy - ey, ox - ex)
    ha = math.asin(min(occluder_radius / dist, 1.0))
    al, ar = ac + ha, ac - ha
    tl = (ox - occluder_radius * math.sin(ac), oy + occluder_radius * math.cos(ac))
    tr = (ox + occluder_radius * math.sin(ac), oy - occluder_radius * math.cos(ac))
    fl = (ox + depth_m * math.cos(al), oy + depth_m * math.sin(al))
    fc = (ox + depth_m * math.cos(ac), oy + depth_m * math.sin(ac))
    fr = (ox + depth_m * math.cos(ar), oy + depth_m * math.sin(ar))
    return [tl, fl, fc, fr, tr]

def build_blind_zone_nodes(frame, max_zones=8, occluder_radius=2.5, depth_m=25.0, max_occ_dist=35.0):
    if frame.ego is None: return []
    ego_xy = (frame.ego.x, frame.ego.y)
    candidates = []
    for obj in frame.objects:
        if obj.object_type not in {"car", "truck", "bus", "vehicle", "parked_vehicle"}: continue
        dist = euclidean(ego_xy, (obj.x, obj.y))
        if dist > max_occ_dist or dist <= occluder_radius: continue
        polygon = compute_blind_zone_polygon(ego_xy, (obj.x, obj.y), occluder_radius, depth_m)
        if not polygon: continue
        cx, cy = polygon_centroid(polygon); area = polygon_area(polygon)
        ac = math.atan2(obj.y - frame.ego.y, obj.x - frame.ego.x)
        candidates.append((dist, ObjectState(
            object_id=f"blind_{obj.object_id}", object_type="occlusion_zone",
            x=cx, y=cy, vx=0., vy=0., heading=ac, visible=False,
            raw={"occluder_id": obj.object_id, "occluder_type": obj.object_type,
                 "occluder_distance": dist, "polygon": polygon,
                 "area": area, "occluder_radius": occluder_radius, "depth_m": depth_m})))
    candidates.sort(key=lambda item: item[0])
    return [node for _, node in candidates[:max_zones]]

def build_blind_labels(scene, frame_index, blind_nodes):
    if scene is None or frame_index is None: return [0] * len(blind_nodes)
    return [build_blind_zone_label(scene, frame_index, node) for node in blind_nodes]

def build_temporal_edges(scene):
    edges = []; previous = {}
    for fi, frame in enumerate(scene.frames):
        nodes = ([frame.ego] if frame.ego else []) + frame.objects
        for ni, node in enumerate(nodes):
            if node.object_id in previous:
                pf, pn = previous[node.object_id]
                edges.append({"source_frame": pf, "source_node": pn,
                               "target_frame": fi, "target_node": ni, "edge_type": "temporal_next"})
            previous[node.object_id] = (fi, ni)
    return edges

def build_frame_graph(frame, neighbor_radius=30.0, scene=None, frame_index=None):
    nodes = ([frame.ego] if frame.ego else []) + list(frame.objects)
    blind_nodes = build_blind_zone_nodes(frame)
    blind_start = len(nodes)
    nodes = nodes + blind_nodes
    node_ids = [n.object_id for n in nodes]
    x = [node_features(n, frame.ego) for n in nodes]
    ei = [[], []]; ea = []; et = []
    for si, src in enumerate(nodes):
        for di, dst in enumerate(nodes):
            if si == di: continue
            d = euclidean((src.x, src.y), (dst.x, dst.y))
            if d > neighbor_radius: continue
            ei[0].append(si); ei[1].append(di)
            ea.append(edge_features(src, dst, d))
            et.append(infer_edge_type(src, dst, d))
    return {"frame_id": frame.frame_id, "timestamp": frame.timestamp,
            "node_ids": node_ids, "node_types": [n.object_type for n in nodes],
            "x": x, "edge_index": ei, "edge_attr": ea, "edge_type": et,
            "y": build_frame_risk_label(frame),
            "blind_node_indices": list(range(blind_start, blind_start + len(blind_nodes))),
            "blind_y": build_blind_labels(scene, frame_index, blind_nodes)}

def build_scene_graph(scene, neighbor_radius=30.0):
    frame_graphs = [
        build_frame_graph(f, neighbor_radius, scene, i)
        for i, f in enumerate(scene.frames)
    ]
    return {"scene_id": scene.scene_id, "source_path": scene.source_path,
            "reference_track": scene.raw.get("reference_track") if isinstance(scene.raw, dict) else None,
            "node_feature_names": NODE_FEATURE_NAMES,
            "edge_feature_names": EDGE_FEATURE_NAMES,
            "y": build_risk_label(scene),
            "frames": frame_graphs,
            "temporal_edges": build_temporal_edges(scene)}

print("✓ 모든 클래스·함수 인라인 정의 완료 (src/ 폴더 불필요)")
print("  ObjectState, Frame, Scene | utils | label_builder | graph_builder")
print(f"  노드 피처: {len(NODE_FEATURE_NAMES)}차원   엣지 피처: {len(EDGE_FEATURE_NAMES)}차원")


In [ ]:
# ── 0-B. 패키지 설치 ──────────────────────────────────────────
!pip install -q shapely networkx matplotlib seaborn pandas numpy tqdm scikit-learn pyarrow
print('패키지 설치 완료')

In [ ]:
# ── 0-C. Google Drive 마운트 ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive 마운트 완료')

In [ ]:
# ── 0-D. Drive → /tmp 복사 (JSON 파싱 10× 속도 향상) ─────────
# ▼▼▼ 본인 Drive 경로로 수정 ▼▼▼
DRIVE_DATA = '/content/drive/MyDrive/그기마(team 6)/imptc_trajectory'
DRIVE_SAVE = '/content/drive/MyDrive/그기마(team 6)/지은'
TMP_DATA   = '/tmp/imptc'

import shutil, os, time

os.makedirs(DRIVE_SAVE, exist_ok=True)

if not os.path.exists(TMP_DATA):
    print('Drive → /tmp 복사 중... (1~3분 소요)')
    t0 = time.time()
    shutil.copytree(DRIVE_DATA, TMP_DATA)
    print(f'복사 완료 ({time.time()-t0:.0f}s)')
else:
    print('/tmp/imptc 이미 존재 — 복사 생략')

for sp in ['train', 'eval', 'test']:
    p = os.path.join(TMP_DATA, sp)
    if os.path.exists(p):
        print(f'  {sp}/  ->  {len(os.listdir(p)):,} tracks')
    else:
        print(f'  {sp}/  없음')

In [ ]:
# ── 1. 임포트 & 상수 ───────────────────────────────────────────
import json, math, pickle, warnings, glob
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns

from shapely.geometry import Point, Polygon, MultiPolygon
from shapely.ops import unary_union
import networkx as nx
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

DATA_PATH = TMP_DATA
SAVE_PATH = DRIVE_SAVE
SPLITS    = ['train', 'eval', 'test']

# IMPTC raw class_name -> 정규화 타입
CLASS_TO_TYPE = {
    'scooter': 'e_scooter', 'bicycle': 'cyclist', 'person': 'pedestrian',
    'vehicle': 'vehicle',   'car': 'vehicle',      'truck': 'truck',
    'bus': 'bus',           'pedestrian': 'pedestrian', 'cyclist': 'cyclist',
    'e_scooter': 'e_scooter', 'wheelchair': 'pedestrian', 'stroller': 'pedestrian',
}

PM_TYPES      = {'e_scooter', 'cyclist'}            # 주 분석 대상
VRU_TYPES     = {'e_scooter', 'cyclist', 'pedestrian'}
VEHICLE_TYPES = {'vehicle', 'truck', 'bus'}

# True: scooter+cyclist만, False: 보행자 포함
PM_ONLY      = True
TARGET_TYPES = PM_TYPES if PM_ONLY else VRU_TYPES

OBS_WINDOW   = 2.0
PRED_HORIZON = 3.0
OCC_RADIUS   = 2.5
BZ_DEPTH     = 25.0
EDGE_DIST    = 15.0
GRID_US      = 40_000
SESSION_GAP  = 300

CLS_COLOR = {
    'vehicle':'#BDC3C7', 'e_scooter':'#E74C3C', 'cyclist':'#F39C12',
    'pedestrian':'#2ECC71', 'truck':'#95A5A6', 'bus':'#7F8C8D', 'unknown':'#AAB7B8',
}

print('임포트 및 상수 설정 완료')
print(f'분석 대상: {TARGET_TYPES}')
print(f'예측 지평선: {PRED_HORIZON}s  |  BZ 깊이: {BZ_DEPTH}m')

In [ ]:
# ── 2-A. track.json 파싱 함수 ─────────────────────────────────
def parse_track(filepath, split):
    """
    track.json 1개 -> dict
    pos_at / vel_at / speed_at: {ts_grid -> value} 고속 조회 인덱스
    """
    track_id = Path(filepath).parent.name
    try:
        data = json.loads(Path(filepath).read_text())
    except Exception:
        return None

    overview    = data.get('overview', {})
    class_name  = overview.get('class_name', 'unknown').lower().strip()
    object_type = CLASS_TO_TYPE.get(class_name, 'unknown')
    track_data  = data.get('track_data', {})
    if not track_data:
        return None

    rows = []
    for frame_idx, frame in track_data.items():
        coords = frame.get('coordinates', [0, 0, 0])
        rows.append({
            'frame_idx': int(frame_idx),
            'ts_us':     int(frame['ts']),
            'ts':        int(frame['ts']) / 1e6,
            'x':         float(coords[0]),
            'y':         float(coords[1]),
            'z':         float(coords[2]) if len(coords) > 2 else 0.0,
            'speed':     float(frame.get('velocity', 0)),
        })

    df = pd.DataFrame(rows).sort_values('ts').reset_index(drop=True)
    dt = df['ts'].diff().clip(lower=1e-4)
    df['vx'] = df['x'].diff().fillna(0) / dt
    df['vy'] = df['y'].diff().fillna(0) / dt
    df['ts_grid'] = (df['ts_us'] / GRID_US).round().astype(int) * GRID_US

    pos_at, vel_at, speed_at = {}, {}, {}
    for row in df.itertuples(index=False):
        tg = row.ts_grid
        if tg not in pos_at:
            pos_at[tg]   = (row.x, row.y)
            vel_at[tg]   = (row.vx, row.vy)
            speed_at[tg] = row.speed

    return {
        'id':          track_id,
        'split':       split,
        'class_name':  class_name,
        'object_type': object_type,
        'ts_min':      df['ts'].min(),
        'ts_max':      df['ts'].max(),
        'ts_grid_set': set(pos_at.keys()),
        'pos_at':      pos_at,
        'vel_at':      vel_at,
        'speed_at':    speed_at,
        'avg_speed':   float(df['speed'].mean()),
        'df':          df,
    }

sample_fp = next(Path(DATA_PATH).glob('**/track.json'))
t = parse_track(str(sample_fp), 'test')
print(f'트랙 ID: {t["id"]},  class: {t["class_name"]} -> {t["object_type"]}')
print(f'타임스탬프: {t["ts_min"]:.0f} ~ {t["ts_max"]:.0f} s')
print(f'프레임: {len(t["df"])},  평균속도: {t["avg_speed"]:.2f} m/s')

In [ ]:
# ── 2-B. 전체 트랙 로드 (/tmp에서 빠르게) ────────────────────
def load_all_tracks(data_path, splits):
    tracks = []
    for split in splits:
        sp = Path(data_path) / split
        if not sp.exists():
            print(f'{split}/ 없음'); continue
        for tid_dir in tqdm(sorted(sp.iterdir()), desc=f'{split:5s}', leave=False):
            fp = tid_dir / 'track.json'
            if not fp.exists(): continue
            tr = parse_track(str(fp), split)
            if tr is not None:
                tracks.append(tr)
    return tracks

print('전체 트랙 로딩 중... (/tmp에서 읽음, 3~5분)')
all_tracks = load_all_tracks(DATA_PATH, SPLITS)

type_cnt  = Counter(t['object_type'] for t in all_tracks)
split_cnt = Counter(t['split']       for t in all_tracks)
print(f'\n로드 완료: {len(all_tracks):,} 트랙')
print('에이전트 타입:')
for ot, cnt in sorted(type_cnt.items(), key=lambda x: -x[1]):
    tag = '<- PM 대상' if ot in PM_TYPES else ('<- VRU' if ot in VRU_TYPES else '')
    print(f'  {ot:<15} {cnt:>5}  {tag}')
print('Split별:', dict(split_cnt))

In [ ]:
# ── 2-C. 세션 그룹화 (타임스탬프 근접성 기반) ────────────────
#
# 핵심 수정: eval 50개 트랙은 65일에 걸쳐 분산되어 있어
# 동시에 존재하는 VRU+Vehicle 조합이 없었음 -> 샘플 0개
# 해결: 전체 트랙 로드 + 시간적으로 겹치는 트랙끼리 세션으로 묶기

def group_sessions(tracks, gap_s=SESSION_GAP):
    """타임스탬프 근접성으로 트랙을 세션으로 묶음."""
    if not tracks:
        return []
    sorted_t    = sorted(tracks, key=lambda t: t['ts_min'])
    sessions    = []
    cur_session = [sorted_t[0]]
    cur_max_ts  = sorted_t[0]['ts_max']

    for tr in sorted_t[1:]:
        if tr['ts_min'] - cur_max_ts <= gap_s:
            cur_session.append(tr)
            cur_max_ts = max(cur_max_ts, tr['ts_max'])
        else:
            sessions.append(cur_session)
            cur_session = [tr]
            cur_max_ts  = tr['ts_max']

    sessions.append(cur_session)
    return sessions


sessions_by_split = {}
for sp in SPLITS:
    sp_tracks = [t for t in all_tracks if t['split'] == sp]
    sessions_by_split[sp] = group_sessions(sp_tracks)

print('세션 그룹화 결과:')
for sp, sessions in sessions_by_split.items():
    if not sessions: continue
    sizes  = [len(s) for s in sessions]
    usable = sum(
        1 for s in sessions
        if any(t['object_type'] in TARGET_TYPES  for t in s)
        and any(t['object_type'] in VEHICLE_TYPES for t in s)
    )
    print(f'  [{sp}] {len(sessions)} 세션  '
          f'(트랙: min={min(sizes)} max={max(sizes)} median={int(np.median(sizes))})  '
          f'PM+Vehicle 가능: {usable}')

all_sessions = []
for sp, sessions in sessions_by_split.items():
    for i, sess in enumerate(sessions):
        all_sessions.append({'split': sp, 'idx': i, 'tracks': sess})
print(f'\n총 세션: {len(all_sessions)}')

In [ ]:
# ── 3-A. EDA: 클래스·속도·트랙길이 분포 ──────────────────────
df_all = pd.concat([
    t['df'].assign(track_id=t['id'], class_name=t['class_name'],
                   object_type=t['object_type'], split=t['split'])
    for t in all_tracks
], ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('IMPTC Dataset EDA', fontsize=14, fontweight='bold')

ax = axes[0]
cc = df_all.groupby('object_type')['track_id'].nunique().sort_values(ascending=False)
colors = ['#E74C3C' if c in PM_TYPES else '#F39C12' if c in VRU_TYPES else '#4A90D9' for c in cc.index]
bars = ax.bar(cc.index, cc.values, color=colors, edgecolor='white')
ax.set_title('Tracks per Class', fontweight='bold')
ax.set_xlabel('Agent Type'); ax.set_ylabel('# Tracks')
ax.tick_params(axis='x', rotation=30)
for bar, v in zip(bars, cc.values):
    ax.text(bar.get_x()+bar.get_width()/2, v+max(cc)*0.01, str(v), ha='center', fontsize=8)
ax.legend(handles=[
    mpatches.Patch(color='#E74C3C', label='PM (분석 대상)'),
    mpatches.Patch(color='#F39C12', label='기타 VRU'),
    mpatches.Patch(color='#4A90D9', label='Vehicle'),
], fontsize=8)

ax = axes[1]
avg_spd = df_all.groupby('track_id').agg(speed=('speed','mean'), ot=('object_type','first')).reset_index()
for ot, c in CLS_COLOR.items():
    sub = avg_spd[avg_spd['ot']==ot]['speed']
    if len(sub): ax.hist(sub, bins=25, color=c, alpha=0.6, label=ot, edgecolor='white')
ax.set_title('Avg Speed per Track [m/s]', fontweight='bold')
ax.set_xlabel('Speed (m/s)'); ax.legend(fontsize=7)

ax = axes[2]
tl = df_all.groupby('track_id').size()
ax.hist(tl, bins=40, color='#27AE60', edgecolor='white', alpha=0.85)
med = tl.median()
ax.axvline(med, color='red', linestyle='--', label=f'Median={med:.0f}')
ax.set_title('Track Length (# Timesteps)', fontweight='bold')
ax.set_xlabel('Timesteps'); ax.legend()

plt.tight_layout()
fig.savefig('/tmp/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'총 관측: {len(df_all):,}  |  트랙: {df_all["track_id"].nunique():,}')

In [ ]:
# ── 3-B. EDA: Top-View 궤적 시각화 ───────────────────────────
import random; random.seed(42)

def plot_trajectories(df, max_tracks=200, title='Trajectories'):
    fig, ax = plt.subplots(figsize=(9, 9))
    groups = [(tid, ot, grp) for (tid, ot), grp in df.groupby(['track_id','object_type'])]
    if len(groups) > max_tracks:
        groups = random.sample(groups, max_tracks)
    for tid, ot, grp in groups:
        grp = grp.sort_values('ts')
        c = CLS_COLOR.get(ot, '#AAB7B8')
        ax.plot(grp['x'].values, grp['y'].values, color=c,
                alpha=0.2 if ot in VEHICLE_TYPES else 0.8,
                lw=0.5    if ot in VEHICLE_TYPES else 1.8)
        if ot in TARGET_TYPES:
            ax.plot(grp['x'].iloc[-1], grp['y'].iloc[-1], 'o', color=c, ms=4, zorder=5)
    legend_h = [Line2D([0],[0], color=c, lw=2, label=ot)
                for ot, c in CLS_COLOR.items() if ot in df['object_type'].unique()]
    ax.legend(handles=legend_h, fontsize=8, loc='upper right')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]')
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
    return fig, ax

fig, ax = plot_trajectories(df_all, title='All Splits Top-View')
plt.tight_layout()
fig.savefig('/tmp/trajectories_all.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3-C. EDA: Split x 에이전트 타입 히트맵 ───────────────────
pivot = df_all.groupby(['split','object_type'])['track_id'].nunique().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns)*1.5), 3))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': '# Tracks'})
ax.set_title('Split x Agent Type  (트랙 수)', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig('/tmp/split_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(pivot)

In [ ]:
# ── 3-D. EDA: 세션 규모 분포 ──────────────────────────────────
sess_rows = []
for si in all_sessions:
    sess = si['tracks']
    n_pm  = sum(1 for t in sess if t['object_type'] in PM_TYPES)
    n_veh = sum(1 for t in sess if t['object_type'] in VEHICLE_TYPES)
    dur   = max(t['ts_max'] for t in sess) - min(t['ts_min'] for t in sess)
    sess_rows.append({'split': si['split'], 'n_tracks': len(sess),
                      'n_pm': n_pm, 'n_vehicle': n_veh, 'duration_s': dur,
                      'usable': int(n_pm > 0 and n_veh > 0)})
df_sess = pd.DataFrame(sess_rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('세션 통계', fontsize=13, fontweight='bold')
axes[0].hist(df_sess['n_tracks'], bins=30, color='#4A90D9', edgecolor='white')
axes[0].set_title('세션 당 트랙 수'); axes[0].set_xlabel('# Tracks')
axes[1].hist(df_sess['duration_s'].clip(0, 600), bins=30, color='#27AE60', edgecolor='white')
axes[1].set_title('세션 지속시간 (초, clip 600s)'); axes[1].set_xlabel('Seconds')
usable = df_sess['usable'].sum()
axes[2].bar(['PM+Vehicle (라벨 가능)', '기타'], [usable, len(df_sess)-usable],
            color=['#E74C3C', '#BDC3C7'])
axes[2].set_title(f'라벨 생성 가능: {usable}/{len(df_sess)}')
plt.tight_layout()
fig.savefig('/tmp/session_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print(df_sess.groupby('split')[['n_tracks','n_pm','n_vehicle','usable']].sum())

In [ ]:
# ── 4-A. Blind Zone 기하 함수 ────────────────────────────────
def compute_blind_zone(ego_pos, occ_pos, occ_r=OCC_RADIUS, depth=BZ_DEPTH):
    """
    Ego 시점에서 occluder 뒤쪽 사각지대 Polygon 반환.
    ego_pos, occ_pos: (x, y) 튜플 [m]
    """
    ex, ey = ego_pos
    ox, oy = occ_pos
    dist = math.hypot(ox - ex, oy - ey)
    if dist <= occ_r:
        return None
    angle_c = math.atan2(oy - ey, ox - ex)
    half_a  = math.asin(min(occ_r / dist, 1.0))
    al, ar  = angle_c + half_a, angle_c - half_a
    tl = (ox - occ_r * math.sin(angle_c), oy + occ_r * math.cos(angle_c))
    tr = (ox + occ_r * math.sin(angle_c), oy - occ_r * math.cos(angle_c))
    fl = (ox + depth * math.cos(al),      oy + depth * math.sin(al))
    fc = (ox + depth * math.cos(angle_c), oy + depth * math.sin(angle_c))
    fr = (ox + depth * math.cos(ar),      oy + depth * math.sin(ar))
    poly = Polygon([tl, fl, fc, fr, tr])
    return poly if poly.is_valid else poly.buffer(0)


def union_blind_zones(ego_pos, occ_positions):
    """여러 occluder BZ 합산 (MultiPolygon 포함 처리)."""
    valid = [compute_blind_zone(ego_pos, p) for p in occ_positions]
    valid = [b for b in valid if b is not None]
    return unary_union(valid) if valid else None


def point_in_bz(bz_union, point):
    if bz_union is None or bz_union.is_empty:
        return False
    return bz_union.contains(Point(point))


bz = compute_blind_zone((0,0), (10,0), 2.5, 20)
assert bz is not None
assert bz.contains(Point(15, 0))
assert not bz.contains(Point(5, 0))
print(f'Blind Zone 기하 테스트 통과  (BZ 면적: {bz.area:.1f} m2)')

In [ ]:
# ── 4-B. Blind Zone 시각화 데모 ───────────────────────────────
np.random.seed(42)
EGO_POS  = np.array([0.0,  0.0])
OCC_POS1 = np.array([12.0, 2.0])
OCC_POS2 = np.array([8.0, -5.0])
VRU_DEMO = [(12 + np.random.randn()*3, 2 + np.random.randn()*4) for _ in range(14)]

def draw_bz_scene(ax, ego, occ_list, vru_pts, title):
    bz_u = union_blind_zones(tuple(ego), [tuple(o) for o in occ_list])
    polys = list(bz_u.geoms) if isinstance(bz_u, MultiPolygon) else [bz_u]
    for i, poly in enumerate(polys):
        xs, ys = poly.exterior.xy
        ax.fill(xs, ys, alpha=0.18, color='#E74C3C')
        ax.plot(xs, ys, 'r--', lw=1.5, label='Blind Zone' if i==0 else '')
    for occ in occ_list:
        ax.add_patch(plt.Circle(occ, OCC_RADIUS, color='#7F8C8D', alpha=0.85, zorder=5))
        ax.text(*occ, 'OCC', ha='center', va='center', fontsize=7, color='white', fontweight='bold', zorder=6)
    ax.plot(*ego, 's', color='#2980B9', ms=12, zorder=7, label='Ego')
    n_hid = 0
    for pos in vru_pts:
        in_bz = point_in_bz(bz_u, pos)
        n_hid += int(in_bz)
        ax.plot(*pos, 'o', color='#E74C3C' if in_bz else '#27AE60', ms=9, zorder=8)
    ax.legend(handles=[
        Line2D([0],[0], marker='s', color='w', markerfacecolor='#2980B9', ms=9, label='Ego'),
        mpatches.Patch(color='#7F8C8D', label='Occluder'),
        mpatches.Patch(color='#E74C3C', alpha=0.5, label='Blind Zone'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor='#E74C3C', ms=8, label='PM (hidden)'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor='#27AE60', ms=8, label='PM (visible)'),
    ], fontsize=8, loc='upper left')
    ax.set_xlim(-5, 42); ax.set_ylim(-15, 15)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.25)
    ax.set_title(f'{title}  (hidden: {n_hid}/{len(vru_pts)})', fontsize=11, fontweight='bold')
    ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
draw_bz_scene(axes[0], EGO_POS, [OCC_POS1],            VRU_DEMO, 'Single Occluder')
draw_bz_scene(axes[1], EGO_POS, [OCC_POS1, OCC_POS2],  VRU_DEMO, 'Multiple Occluders (Union BZ)')
plt.suptitle('Blind Zone Demo', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig('/tmp/blind_zone_demo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5. 세션 -> Scene 변환 ────────────────────────────────────────────

def session_to_scene(session_tracks, session_id, stride=5):
    """
    세션 트랙 목록 -> Scene
    stride: 타임스탬프 다운샘플링 (속도 최적화)
    """
    all_ts_grid = sorted(set(
        ts for track in session_tracks for ts in track['ts_grid_set']
    ))
    frames = []
    for i, ts_g in enumerate(all_ts_grid):
        if i % stride != 0:
            continue
        active = [(tr, tr['pos_at'][ts_g], tr['vel_at'].get(ts_g, (0.0, 0.0)))
                  for tr in session_tracks if ts_g in tr['pos_at']]
        if not active:
            continue
        ego_obj  = None
        best_spd = -1.0
        objects  = []
        for track, (x, y), (vx, vy) in active:
            otype   = track['object_type']
            speed   = track['speed_at'].get(ts_g, 0.0)
            heading = math.atan2(vy, vx) if (abs(vx) + abs(vy)) > 0.01 else 0.0
            obj = ObjectState(object_id=track['id'], object_type=otype,
                              x=x, y=y, vx=vx, vy=vy, heading=heading, visible=True)
            if otype in VEHICLE_TYPES and speed > best_spd:
                if ego_obj is not None:
                    objects.append(ego_obj)
                ego_obj  = obj
                best_spd = speed
            else:
                objects.append(obj)
        frames.append(Frame(frame_id=str(ts_g), timestamp=ts_g / 1e6,
                            ego=ego_obj, objects=objects))
    return Scene(scene_id=session_id, frames=frames)


test_sess  = max(all_sessions, key=lambda s: len(s['tracks']))
test_scene = session_to_scene(
    test_sess['tracks'], f"{test_sess['split']}_sess{test_sess['idx']}")
print(f'테스트 세션: {len(test_sess["tracks"])} 트랙 -> {len(test_scene.frames)} 프레임')
if test_scene.frames:
    f0 = test_scene.frames[0]
    print(f'첫 프레임: ego={f0.ego and f0.ego.object_type}, objects={len(f0.objects)}')


In [ ]:
# ── 6-A. Emergence 라벨 생성 ─────────────────────────────────

def generate_labels_for_session(session_tracks, session_id,
                                 obs_s=OBS_WINDOW, pred_s=PRED_HORIZON,
                                 occ_r=OCC_RADIUS, bz_d=BZ_DEPTH, stride=5):
    """
    한 세션 -> blind-zone emergence 샘플 리스트
    label=1: pred_s 초 이내에 사각지대 밖으로 출현
    label=0: pred_s 초 동안 계속 숨겨져 있음
    """
    samples = []
    veh_tracks = [t for t in session_tracks if t['object_type'] in VEHICLE_TYPES]
    pm_tracks  = [t for t in session_tracks if t['object_type'] in TARGET_TYPES]
    if not veh_tracks or not pm_tracks:
        return samples

    all_ts_grid = sorted(set(
        ts for t in session_tracks for ts in t['ts_grid_set']
    ))
    if len(all_ts_grid) < 2:
        return samples

    grid_step_s = GRID_US / 1e6
    obs_steps   = max(1, int(obs_s  / grid_step_s))
    pred_steps  = max(1, int(pred_s / grid_step_s))

    for i in range(obs_steps, len(all_ts_grid) - pred_steps, stride):
        t_now = all_ts_grid[i]

        active_veh = [(t, t['pos_at'][t_now], t['speed_at'].get(t_now, 0))
                      for t in veh_tracks if t_now in t['pos_at']]
        if not active_veh:
            continue
        ego_tr, ego_pos, _ = max(active_veh, key=lambda x: x[2])

        occ_positions = [pos for tr, pos, spd in active_veh
                         if tr['id'] != ego_tr['id'] and spd < 3.0]
        if not occ_positions:
            occ_positions = [pos for tr, pos, spd in active_veh if tr['id'] != ego_tr['id']]
        if not occ_positions:
            continue

        bz_union = union_blind_zones(ego_pos, occ_positions)
        if bz_union is None or bz_union.is_empty:
            continue

        future_ts = all_ts_grid[i:i + pred_steps]

        for pm_tr in pm_tracks:
            if t_now not in pm_tr['pos_at']:
                continue
            pm_pos = pm_tr['pos_at'][t_now]
            if not point_in_bz(bz_union, pm_pos):
                continue

            label = int(any(
                (fp := pm_tr['pos_at'].get(ft)) is not None
                and not point_in_bz(bz_union, fp)
                for ft in future_ts
            ))

            samples.append({
                'session_id':   session_id,
                'split':        pm_tr['split'],
                'ego_id':       ego_tr['id'],
                'pm_id':        pm_tr['id'],
                'pm_class':     pm_tr['object_type'],
                't_now_us':     t_now,
                'ego_pos':      ego_pos,
                'pm_pos':       pm_pos,
                'occ_pos_list': occ_positions,
                'label':        label,
            })
    return samples


usable_sessions = [
    s for s in all_sessions
    if any(t['object_type'] in TARGET_TYPES  for t in s['tracks'])
    and any(t['object_type'] in VEHICLE_TYPES for t in s['tracks'])
]
print(f'라벨 생성 대상 세션: {len(usable_sessions)} / {len(all_sessions)}')

all_samples = []
for sess_info in tqdm(usable_sessions, desc='세션 처리'):
    sid = f"{sess_info['split']}_sess{sess_info['idx']}"
    s   = generate_labels_for_session(sess_info['tracks'], sid)
    all_samples.extend(s)

n_pos = sum(x['label'] for x in all_samples)
n_neg = len(all_samples) - n_pos
print(f'\n총 샘플: {len(all_samples):,}')
print(f'  Positive (출현): {n_pos:,}  ({100*n_pos/max(len(all_samples),1):.1f}%)')
print(f'  Negative (숨김): {n_neg:,}  ({100*n_neg/max(len(all_samples),1):.1f}%)')
if all_samples:
    print('PM 타입별:', dict(Counter(s['pm_class'] for s in all_samples)))

In [ ]:
# ── 6-B. 데이터셋 통계 시각화 ────────────────────────────────
if not all_samples:
    print('샘플 없음. 세션 그룹화 확인 또는 OCC_RADIUS/BZ_DEPTH 조정 필요.')
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Blind-Zone Emergence 데이터셋 통계', fontsize=14, fontweight='bold')

    labels  = [s['label']    for s in all_samples]
    classes = [s['pm_class'] for s in all_samples]
    splits  = [s['split']    for s in all_samples]
    n_p, n_n = sum(labels), len(labels)-sum(labels)

    ax = axes[0][0]
    ax.bar(['Negative (숨김)', 'Positive (출현)'], [n_n, n_p],
           color=['#3498DB', '#E74C3C'], edgecolor='white')
    ax.set_title('라벨 분포', fontweight='bold')
    for i, (cnt, tot) in enumerate([(n_n, len(labels)), (n_p, len(labels))]):
        ax.text(i, cnt+max(n_p,n_n)*0.02, f'{cnt:,}\n({100*cnt/tot:.1f}%)',
                ha='center', fontsize=10, fontweight='bold')

    ax = axes[0][1]
    cc2 = Counter(classes)
    ax.bar(cc2.keys(), cc2.values(), color=sns.color_palette('Set2', len(cc2)))
    ax.set_title('PM 클래스별 샘플 수', fontweight='bold')
    ax.tick_params(axis='x', rotation=25)

    ax = axes[1][0]
    pos_by_cls = defaultdict(list)
    for s in all_samples: pos_by_cls[s['pm_class']].append(s['label'])
    cls_names = list(pos_by_cls.keys())
    pos_rates = [np.mean(pos_by_cls[c])*100 for c in cls_names]
    bars = ax.bar(cls_names, pos_rates, color=sns.color_palette('Set1', len(cls_names)))
    ax.set_title('PM 클래스별 Positive Rate (%)', fontweight='bold')
    ax.set_ylim(0, 105); ax.tick_params(axis='x', rotation=25)
    for bar, v in zip(bars, pos_rates):
        ax.text(bar.get_x()+bar.get_width()/2, v+1, f'{v:.1f}%', ha='center', fontsize=9)

    ax = axes[1][1]
    sp_cnt = Counter(splits)
    ax.bar(sp_cnt.keys(), sp_cnt.values(), color=sns.color_palette('Paired', len(sp_cnt)))
    ax.set_title('Split별 샘플 수', fontweight='bold')

    plt.tight_layout()
    fig.savefig('/tmp/dataset_stats.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 6-C. Emergence 이벤트 시각화 ─────────────────────────────
track_dict = {t['id']: t for t in all_tracks}

def visualize_emergence(sample, track_dict, n_steps=3):
    pm_tr = track_dict.get(sample['pm_id'])
    if pm_tr is None: return
    bz_union = union_blind_zones(sample['ego_pos'], sample['occ_pos_list'])
    fig, axes = plt.subplots(1, n_steps, figsize=(6*n_steps, 6))

    for ax, t_off in zip(axes, np.linspace(0, PRED_HORIZON, n_steps)):
        if bz_union:
            polys = list(bz_union.geoms) if isinstance(bz_union, MultiPolygon) else [bz_union]
            for poly in polys:
                xs, ys = poly.exterior.xy
                ax.fill(xs, ys, alpha=0.15, color='#E74C3C')
                ax.plot(xs, ys, 'r--', lw=1.3)
        for occ_pos in sample['occ_pos_list']:
            ax.add_patch(plt.Circle(occ_pos, OCC_RADIUS, color='#7F8C8D', alpha=0.85, zorder=5))
        ax.plot(*sample['ego_pos'], 's', color='#2980B9', ms=11, zorder=7, label='Ego')

        t_now = sample['t_now_us']
        past_ts = t_now - int(OBS_WINDOW*1e6)
        traj = [(pos[0], pos[1]) for ts_g, pos in sorted(pm_tr['pos_at'].items())
                if past_ts <= ts_g <= t_now]
        if traj:
            tx, ty = zip(*traj)
            ax.plot(tx, ty, 'g-', alpha=0.4, lw=1.5)

        target  = t_now + int(t_off*1e6)
        nearest = min(pm_tr['pos_at'], key=lambda ts: abs(ts-target), default=None)
        if nearest and abs(nearest-target) < GRID_US*5:
            pmx, pmy = pm_tr['pos_at'][nearest]
            in_bz = point_in_bz(bz_union, (pmx, pmy))
            ax.plot(pmx, pmy, 'o', color='#E74C3C' if in_bz else '#27AE60', ms=13, zorder=9,
                    label=f'PM +{t_off:.1f}s ({"BZ" if in_bz else "출현"})')

        ax.set_title(f't = +{t_off:.1f}s', fontsize=10, fontweight='bold')
        ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]')
        ax.set_aspect('equal'); ax.grid(True, alpha=0.25)
        ax.legend(fontsize=7, loc='upper left')

    lbl_str = f'{"출현" if sample["label"] else "미출현"}  (label={sample["label"]})'
    plt.suptitle(f'{sample["session_id"]} | PM={sample["pm_id"]} ({sample["pm_class"]}) | {lbl_str}',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    fig.savefig(f'/tmp/emergence_{sample["pm_id"]}.png', dpi=150, bbox_inches='tight')
    plt.show()


pos_samples = [s for s in all_samples if s['label']==1]
neg_samples = [s for s in all_samples if s['label']==0]
print(f'Positive: {len(pos_samples):,}   Negative: {len(neg_samples):,}')
if pos_samples: visualize_emergence(pos_samples[0], track_dict)
if neg_samples: visualize_emergence(neg_samples[0], track_dict)

In [ ]:
# ── 7-A. 그래프 구성 ────────────────────────────────────────────────

rich_sessions = [
    s for s in all_sessions
    if any(t['object_type'] in TARGET_TYPES  for t in s['tracks'])
    and any(t['object_type'] in VEHICLE_TYPES for t in s['tracks'])
]
print(f'그래프 구성 대상 세션: {len(rich_sessions):,}')

demo_sess  = rich_sessions[0]
demo_sid   = f"{demo_sess['split']}_sess{demo_sess['idx']}"
demo_scene = session_to_scene(demo_sess['tracks'], demo_sid, stride=10)
print(f'\n데모: {demo_sid}  ({len(demo_sess["tracks"])} 트랙, {len(demo_scene.frames)} 프레임)')

if demo_scene.frames:
    demo_graph = build_scene_graph(demo_scene, neighbor_radius=EDGE_DIST)
    f0 = demo_graph['frames'][0]
    print(f'\n그래프 구조:')
    print(f'  scene_id: {demo_graph["scene_id"]}')
    print(f'  프레임 수: {len(demo_graph["frames"])}')
    print(f'  첫 프레임 노드: {len(f0["node_ids"])}  (blind_zone: {len(f0["blind_node_indices"])})')
    print(f'  첫 프레임 엣지: {len(f0["edge_index"][0])}')
    print(f'  엣지 타입: {Counter(f0["edge_type"]).most_common()}')
    print(f'  노드 피처 dim: {len(f0["x"][0]) if f0["x"] else "N/A"}')
    print(f'  엣지 피처 dim: {len(f0["edge_attr"][0]) if f0["edge_attr"] else "N/A"}')


In [ ]:
# ── 7-B. NetworkX 그래프 시각화 ───────────────────────────────
if not demo_scene.frames:
    print('프레임 없음 — 스킵')
else:
    best_frame = max(demo_graph['frames'], key=lambda f: len(f['node_ids']))
    G = nx.DiGraph()

    for nidx, (nid, ntype, feat) in enumerate(
        zip(best_frame['node_ids'], best_frame['node_types'], best_frame['x'])
    ):
        G.add_node(nidx, label=f'{nid[:8]}\n{ntype[:6]}', ntype=ntype, pos=(feat[0], feat[1]))

    for src, dst, etype in zip(
        best_frame['edge_index'][0], best_frame['edge_index'][1], best_frame['edge_type']
    ):
        G.add_edge(src, dst, etype=etype)

    NODE_CLR = {
        'ego_vehicle':'#2980B9', 'e_scooter':'#E74C3C', 'cyclist':'#F39C12',
        'pedestrian':'#2ECC71', 'vehicle':'#BDC3C7', 'truck':'#95A5A6',
        'bus':'#7F8C8D', 'occlusion_zone':'#8E44AD', 'unknown':'#AAB7B8',
    }
    EDGE_CLR = {
        'spatial_near':'#BDC3C7', 'potential_conflict':'#E74C3C',
        'occludes':'#8E44AD', 'blind_zone_relation':'#9B59B6',
    }

    nc = [NODE_CLR.get(G.nodes[n]['ntype'], '#AAB7B8') for n in G.nodes]
    ec = [EDGE_CLR.get(G.edges[e]['etype'], '#BDC3C7') for e in G.edges]

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    pos_dict = {n: G.nodes[n]['pos'] for n in G.nodes}
    nx.draw_networkx(G, pos=pos_dict, ax=axes[0],
                     node_color=nc, edge_color=ec, node_size=300,
                     labels={n: G.nodes[n]['label'] for n in G.nodes},
                     font_size=6, arrows=True, arrowsize=10, alpha=0.9, width=1.5)
    axes[0].set_title('이종 장면 그래프 (실제 좌표)', fontsize=11, fontweight='bold')
    axes[0].set_xlabel('X [m]'); axes[0].set_ylabel('Y [m]'); axes[0].grid(True, alpha=0.2)

    pos_spring = nx.spring_layout(G, seed=42)
    nx.draw_networkx(G, pos=pos_spring, ax=axes[1],
                     node_color=nc, edge_color=ec, node_size=400,
                     labels={n: G.nodes[n]['label'] for n in G.nodes},
                     font_size=6, arrows=True, arrowsize=12, alpha=0.9, width=2.0)
    axes[1].set_title('이종 장면 그래프 (Spring Layout)', fontsize=11, fontweight='bold')

    legend_handles = (
        [mpatches.Patch(color=c, label=nt) for nt, c in NODE_CLR.items()
         if any(G.nodes[n]['ntype']==nt for n in G.nodes)] +
        [Line2D([0],[0], color=c, lw=2, label=et) for et, c in EDGE_CLR.items()
         if any(G.edges[e]['etype']==et for e in G.edges)]
    )
    axes[1].legend(handles=legend_handles, fontsize=8, loc='upper right', framealpha=0.9)

    plt.suptitle(
        f'Scene Graph | {demo_sid} | frame={best_frame["frame_id"]} | '
        f'nodes={G.number_of_nodes()} edges={G.number_of_edges()}',
        fontsize=12, fontweight='bold')
    plt.tight_layout()
    fig.savefig('/tmp/scene_graph.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'노드: {G.number_of_nodes()}   엣지: {G.number_of_edges()}')
    print(f'엣지 타입: {Counter(d["etype"] for _,_,d in G.edges(data=True)).most_common()}')

In [ ]:
# ── 7-C. 전체 그래프 데이터셋 구성 ───────────────────────────
graph_dataset = []
node_feat_dim = None
edge_feat_dim = None

for sess_info in tqdm(rich_sessions, desc='그래프 구성'):
    sid   = f"{sess_info['split']}_sess{sess_info['idx']}"
    scene = session_to_scene(sess_info['tracks'], sid, stride=10)
    if not scene.frames: continue
    graph = build_scene_graph(scene, neighbor_radius=EDGE_DIST)
    graph_dataset.append(graph)
    if node_feat_dim is None and graph['frames']:
        f0 = graph['frames'][0]
        if f0['x']:         node_feat_dim = len(f0['x'][0])
        if f0['edge_attr']: edge_feat_dim = len(f0['edge_attr'][0])

total_frames = sum(len(g['frames']) for g in graph_dataset)
total_nodes  = sum(len(f['node_ids']) for g in graph_dataset for f in g['frames'])
total_edges  = sum(len(f['edge_index'][0]) for g in graph_dataset for f in g['frames'])
n_pos_scenes = sum(g['y'] for g in graph_dataset)

print(f'그래프 데이터셋 완료')
print(f'  씬:    {len(graph_dataset):,}')
print(f'  프레임: {total_frames:,}')
print(f'  노드:   {total_nodes:,}')
print(f'  엣지:   {total_edges:,}')
print(f'  노드 피처 dim: {node_feat_dim}')
print(f'  엣지 피처 dim: {edge_feat_dim}')
print(f'  Positive 씬: {n_pos_scenes} / {len(graph_dataset)}')

In [ ]:
# ── 8. 결과 저장 (Drive) ──────────────────────────────────────
import shutil
os.makedirs(SAVE_PATH, exist_ok=True)

with open('/tmp/samples.pkl', 'wb') as f:
    pickle.dump(all_samples, f)
shutil.copy('/tmp/samples.pkl', f'{SAVE_PATH}/samples.pkl')
print(f'samples.pkl  ({len(all_samples):,} 샘플)')

if all_samples:
    df_summary = pd.DataFrame([{
        'session_id': s['session_id'], 'split': s['split'],
        'ego_id': s['ego_id'], 'pm_id': s['pm_id'],
        'pm_class': s['pm_class'], 't_now_us': s['t_now_us'],
        'n_occluders': len(s['occ_pos_list']), 'label': s['label'],
    } for s in all_samples])
    df_summary.to_csv(f'{SAVE_PATH}/samples_summary.csv', index=False)
    print(f'samples_summary.csv  ({len(df_summary)} rows)')

df_all.to_parquet(f'{SAVE_PATH}/all_tracks.parquet', index=False)
print(f'all_tracks.parquet  ({len(df_all):,} rows)')

with open('/tmp/graph_dataset.pkl', 'wb') as f:
    pickle.dump(graph_dataset, f)
shutil.copy('/tmp/graph_dataset.pkl', f'{SAVE_PATH}/graph_dataset.pkl')
print(f'graph_dataset.pkl  ({len(graph_dataset)} 씬)')

for png in glob.glob('/tmp/*.png'):
    shutil.copy(png, f"{SAVE_PATH}/{os.path.basename(png)}")

print(f'\n=== 저장 완료: {SAVE_PATH} ===')
for fn in sorted(os.listdir(SAVE_PATH)):
    sz = os.path.getsize(os.path.join(SAVE_PATH, fn)) / 1024 / 1024
    print(f'  {fn:<40} ({sz:.2f} MB)')

---
## 다음 단계 — 모델 학습

```python
import pickle, pandas as pd

with open('samples.pkl', 'rb') as f:
    samples = pickle.load(f)      # emergence 라벨 샘플

with open('graph_dataset.pkl', 'rb') as f:
    graphs = pickle.load(f)       # scene graph dicts

df_all = pd.read_parquet('all_tracks.parquet')
```

### 모델 로드맵

| 단계 | 모델 | 비고 |
|------|------|---------|
| Baseline | 거리·속도 규칙 | ego <-> BZ 거리 스코어 |
| MLP | node feat flatten | 빠른 검증 |
| **GraphSAGE** | HeteroData PyG | 이종 그래프 |
| **ST-GNN+GRU** | 시공간 그래프 | GNN per frame -> GRU |

### 노드 피처 (11차원)
```
[x, y, vx, vy, heading, type_id, dist_to_ego, rel_angle, visible, is_occluder, is_vru]
```

### 엣지 피처 (6차원)
```
[distance, rel_vx, rel_vy, rel_heading, TTC, visibility_blocked]
```

### 평가 지표
- **AUROC / AUPRC**: 클래스 불균형 대응
- **Recall @ high Precision**: false negative 최소화 (safety-critical)
- **F1 Score**: 종합 성능